# W03 除錯挑戰｜從「能跑」到「能被驗證」

**人工智慧於醫學影像組學的分析與應用（3010040）｜預計 17 分鐘**

---

## 學習目標

辨認一份「執行不報錯」的 notebook 為何仍可能無法交付，並把它修成可攜、可重建、可重現的分析流程。

### 前提

這份 notebook 執行完全正常，**不會出現語法錯誤**。不要把「沒有錯誤訊息」誤認為「研究流程正確」。

### 提示

三個問題分別藏在 **版本、路徑、隨機** 附近。每個問題都要同時寫出：

1. 問題位於哪個儲存格；
2. 它會造成什麼後果；
3. 你打算如何驗證修補有效。

### 課堂流程

1. **教師說明（2 分鐘）**：確認任務與提示。
2. **個人檢視（5 分鐘）**：先獨立找問題，找到一個就先記錄。
3. **配對比對（5 分鐘）**：針對不一致的判斷，用後果與證據說服對方。
4. **合作修補（4 分鐘）**：駕駛操作、領航員檢查，兩人都要能解釋修改。
5. **全班回收（1 分鐘）**：分享最難發現的問題。

### 完成判準

> 把修補後的 notebook 交給另一組，對方**不問任何問題**就能執行，且從頭重跑兩次會得到**相同數字**。

把找到的問題記在這裡（雙擊儲存格即可編輯）：

| # | 問題在哪一格 | 會造成什麼後果 | 如何驗證修補有效 |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

## 1. 安裝

In [ ]:
# 安裝需要的套件
!pip install -q scikit-learn pandas numpy matplotlib

## 2. 匯入與設定輸出位置

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 輸出位置
OUT_DIR = "/content/w03_output"
os.makedirs(OUT_DIR, exist_ok=True)
print("輸出會存到：", OUT_DIR)

## 3. 載入資料

In [ ]:
# 模擬「120 例已萃取好特徵的 CT 影像資料」
# （為了讓這本 notebook 在任何機器上都跑得動，這裡用合成資料代替真實特徵表）
X, y = make_classification(
    n_samples=120, n_features=30, n_informative=4, n_redundant=5,
    class_sep=1.0, weights=[0.65, 0.35], random_state=2026,
)
feat_names = [f"feature_{i:02d}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feat_names)
df["label"] = y
df.insert(0, "subject", [f"SUB{i+1:03d}" for i in range(len(df))])
print(df.shape)
df.head()

## 4. 切分

In [ ]:
# 切分訓練 / 測試
X_train, X_test, y_train, y_test = train_test_split(
    df[feat_names], df["label"], test_size=0.3, stratify=df["label"]
)
print("train", X_train.shape, "test", X_test.shape)

## 5. 建模與評估

In [ ]:
# 建模與評估
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"測試集 AUC = {auc:.3f}")

## 6. 輸出

In [ ]:
# 輸出結果
result = pd.DataFrame({"metric": ["AUC"], "value": [auc]})
result.to_csv(os.path.join(OUT_DIR, "result.csv"), index=False)

imp = pd.DataFrame({"feature": feat_names, "importance": model.feature_importances_})
imp.sort_values("importance", ascending=False).to_csv(
    os.path.join(OUT_DIR, "feature_importance.csv"), index=False)

print("已輸出：", os.listdir(OUT_DIR))

---
### 還沒頭緒的話，試一件事

從最上面**把整本再跑一次**，把兩次的 AUC 都記下來。

| 第幾次 | AUC |
|---|---|
| 第一次 |  |
| 第二次 |  |

如果兩次不一樣 —— 為什麼？同一份資料、同一個模型、同一段程式。

---
## 修補後驗收

不要只確認「現在這次有跑完」。請依序完成：

- [ ] 重新啟動 runtime，從第一格執行到最後一格
- [ ] 再從頭執行一次，記錄兩次 AUC，確認數值相同
- [ ] 確認輸出存在可持久保存的位置，而不是會隨 runtime 消失的暫存空間
- [ ] 確認另一位同學不需要詢問套件版本、資料位置或執行順序
- [ ] 由配對夥伴說明三項修補各自避免了什麼研究風險

### 延伸觀察（不列入本挑戰的三個答案）

輸出檔固定叫 `result.csv`，每次都會覆寫。依課程命名約定，正式研究應加入 `YYYYMMDD`、分析內容與重要參數，保留每次實驗紀錄。

> 本練習只使用合成資料與 `SUB` 代號；真實醫療資料還必須完成 DICOM 標頭、影像像素與身分對照表的去識別化檢核。